## 1. Kirjastot

In [1]:
import keras
from keras.datasets import imdb
from keras.layers import (
    Layer, Embedding, Input, Dense,
    GlobalAveragePooling1D, Dropout, MultiHeadAttention,
    Add, LayerNormalization
)
from keras.models import Model

## 2. IMDB-aineisto

In [2]:
max_features = 10000  # sanaston koko
max_len = 250       # arvostelun pituus tokeneina

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

x_train = keras.utils.pad_sequences(x_train, maxlen=max_len, padding="post")
x_test = keras.utils.pad_sequences(x_test, maxlen=max_len, padding="post")

print("Koulutusaineisto:", x_train.shape, y_train.shape)
print("Testiaineisto:", x_test.shape, y_test.shape)

Koulutusaineisto: (25000, 250) (25000,)
Testiaineisto: (25000, 250) (25000,)


## 3. Token- ja sijaintiembedding

In [3]:
class TokenAndPositionEmbedding(Layer):
    def __init__(self, seq_len, vocab_size, emb_dim):
        super().__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=emb_dim)
        self.pos_emb = Embedding(input_dim=seq_len, output_dim=emb_dim)

    def call(self, x_input):
        seq_len = keras.ops.shape(x_input)[-1]
        positions = keras.ops.arange(start=0, stop=seq_len, step=1)
        positions = self.pos_emb(positions)
        x_input = self.token_emb(x_input)
        return x_input + positions

## 4. Attention-lähtömalli


In [4]:
embed_dim = 32
num_heads = 2
key_dim = embed_dim // num_heads

inputs = Input(shape=(max_len,), dtype="int32")
x = TokenAndPositionEmbedding(max_len, max_features, embed_dim)(inputs)
attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)

x = GlobalAveragePooling1D()(attention_output)
x = Dropout(0.5)(x)
outputs = Dense(1, activation="sigmoid")(x)
att_model = Model(inputs=inputs, outputs=outputs)

att_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

2026-11-16 17:07:10.397656: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-11-16 17:07:10.397674: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-11-16 17:07:10.397678: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1794841630.397686 3331197 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1794841630.397699 3331197 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## 5. Residual ja LayerNormalization lisäykset malliin

In [5]:
embed_dim = 32
num_heads = 2
key_dim = embed_dim // num_heads

inputs = Input(shape=(max_len,), dtype="int32")

# Embedding
x = TokenAndPositionEmbedding(max_len, max_features, embed_dim)(inputs)

# MultiHeadAttention
attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)

# 1. residual connection
x = Add()([x, attention_output])

# 1. LayerNormalization
x = LayerNormalization()(x)

# Feedforward Network
ffn_output = Dense(64, activation="relu")(x)
ffn_output = Dense(embed_dim)(ffn_output)

# 2. residual connection
x = Add()([x, ffn_output])

# 2. LayerNormalization
x = LayerNormalization()(x)

# Luokittelu
x = GlobalAveragePooling1D()(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation="sigmoid")(x)

# Mallin luonti ja compile
att_model_residual_layernorm = Model(inputs=inputs, outputs=outputs)

att_model_residual_layernorm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [6]:
att_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 250)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 250, 32)   │    328,000 │ input_layer[0][0] │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 250, 32)   │      4,224 │ token_and_positi… │
│ (MultiHeadAttentio… │                   │            │ token_and_positi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ multi_head_atten… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │         33 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 332,257 (1.27 MB)

 Trainable params: 332,257 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
att_model_residual_layernorm.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 250)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 250, 32)   │    328,000 │ input_layer_1[0]… │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 250, 32)   │      4,224 │ token_and_positi… │
│ (MultiHeadAttentio… │                   │            │ token_and_positi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 250, 32)   │          0 │ token_and_positi… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 250, 32)   │         64 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 250, 64)   │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 250, 32)   │      2,080 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 250, 32)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 250, 32)   │         64 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 32)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 336,577 (1.28 MB)

 Trainable params: 336,577 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

## 6. Koulutus

In [8]:
history_att = att_model.fit(x_train, y_train, epochs=5, batch_size=32)

history_layer = att_model_residual_layernorm.fit(x_train, y_train, epochs=5, batch_size=32)

Epoch 1/5


2026-11-16 17:07:10.879958: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


782/782 ━━━━━━━━━━━━━━━━━━━━ 26s 32ms/step - accuracy: 0.7826 - loss: 0.4274
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 25s 32ms/step - accuracy: 0.9081 - loss: 0.2331
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.9345 - loss: 0.1752
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 25s 31ms/step - accuracy: 0.9513 - loss: 0.1346
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 25s 31ms/step - accuracy: 0.9619 - loss: 0.1056
Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 33s 41ms/step - accuracy: 0.8071 - loss: 0.4032
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 32s 41ms/step - accuracy: 0.9148 - loss: 0.2208
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 32s 41ms/step - accuracy: 0.9405 - loss: 0.1660
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 32s 41ms/step - accuracy: 0.9560 - loss: 0.1289
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 32s 40ms/step - accuracy: 0.9676 - loss: 0.0993


## 7. Testaus

In [9]:
test_loss_att, test_accuracy_att = att_model.evaluate(x_test, y_test)
print(f"Test accuracy only att = {test_accuracy_att:.4f}")
print(f"Test loss only att = {test_loss_att:.4f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.8555 - loss: 0.4845
Test accuracy only att = 0.8555
Test loss only att = 0.4845


In [10]:
test_loss_att_model_residual_layernorm, test_accuracy_att_model_residual_layernorm = att_model_residual_layernorm.evaluate(x_test, y_test)
print(f"Test accuracy layered = {test_accuracy_att_model_residual_layernorm:.4f}")
print(f"Test loss layered = {test_loss_att_model_residual_layernorm:.4f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.8501 - loss: 0.4804
Test accuracy layered = 0.8501
Test loss layered = 0.4804
